# Machine learning potentials for reaction optimization

Machine learning potentials have already replaced DFT calculations for routine conformer optimization and ranking. Now they are also starting to replace DFT for reaction simulations. In this tutorial, we will investigate how they work for two traditionally challenging reaction types:
- An S<sub>N</sub>2 reaction with a charged nucleophile and leaving group
- Oxidative addition of bromobenzene to Pd(0)

We will see how far the MLPs have evolved, comparing three types:
- "First-generation" foundational model trained mostly on materials data: MACE-MP-0
- "First-generation" foundational model trained on organic conformer data: MACE-OFF
- "Second-generation" foundational model trained on the OMol dataset: OrbMol v3 Conservative 
- "Third-generation" foundational model trained on the OMol dataset with explicit treatment of spin and electrostatics: MACE-Polar-1

Due to computational constraints, we will mostly use the smaller types of these models to allow you to complete the tutorial on time. If you have access to a GPU, you can run the larger models.

## Imports

Most MLPs have interfaces to the Atomic Simulation Environment, a Python package for atomistic simulations with a focus on materials simulations. You can also use MLPs through ORCA, which is the recommended path if you have less experience of coding. Another alternative is to run MLPs on the [Rowan Scientific](https://rowansci.com) platform if you just want to do some quicker calculations. Here, we are going to use ASE due to constraints of having to run everything locally and without an ORCA license.

We will make use of the following packages:
- [ase](https://ase-lib.org) which serves as the basic infrastructure that we build on and do some of the calculations.
- [mlfsm](https://github.com/thegomeslab/ML-FSM) which we will use to find approximate reaction paths that allow us to locate a guess of the transition state
- [sella](https://github.com/zadorlab/sella) which allows us to fully optimize the transition states and also to calculate the intrinsic reaction coordinate to check whether we have actually found the right TS
- [orb-models](https://github.com/orbital-materials/orb-models) which gives us access to the MLP models from Orbital Materials
- [mace](https://github.com/ACEsuit/mace) which gives us access to the MACE family of MLP models

Finally, we have made some functions in `utils.py` to facilitate the calculations, without overburdening you with details. You can always go to this file to investigate more in detail if you are interested.

You might get a message: `cuequivariance or cuequivariance_torch is not available`. You can safely ignore that.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ase.io import read, write
from mace.calculators import mace_mp, mace_off, mace_polar
from mlfsm.cos import FreezingString
from mlfsm.geom import calculate_arc_length, project_trans_rot
from mlfsm.opt import CartesianOptimizer
from orb_models.forcefield import pretrained
from orb_models.forcefield.inference.calculator import ORBCalculator
from sella import Sella

import utils

importlib.reload(utils)
from utils import (
    EV_TO_KCAL_MOL,
    draw_pes_meshgrid,
    ordered_irc_path,
    relaxed_2d_scan,
    reoptimize_ts,
    snapshot_row,
)

## Nucleophilic substitution

Here, we will look at a very simple nucleophilic substitution reaction where a fluoride anion attacks chloromethane.

![sn2 reaction](data/drawings/sn2.png)

We will use the MACE POLAR-1-S model as a baseline for this, and compare it to other MLPs and also DFT. We start by setting up the directories, atom indices for the reactive atoms and the charge and multiplicity.

In [ ]:
DEVICE = "cpu"  # You can change this if you have a GPU available
calc_polar_1_s = mace_polar(model="polar-1-s", default_dtype="float64", device=DEVICE)

SN2_DATA_DIR = Path("data") / "sn2"
SN2_OUTPUT_DIR = Path("output") / "sn2"
SN2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Atom order in SN2 files: C, H, H, H, Cl, F
C_IDX, CL_IDX, F_IDX = 0, 4, 5

# SN2 reaction constants
SN2_CHARGE = -1
SN2_MULTIPLICITY = 1

## Freezing string method

We have already optimized the reactant and product complexes for you, so we will load the corresponding xyz files and use the `project_trans_rot` function in ML-FSM to align them for performing the Freezing String Method for locating an approximate reaction path and guess transition state.

In [ ]:
reactant = read(SN2_DATA_DIR / "reactant_complex.xyz")
product = read(SN2_DATA_DIR / "product_complex.xyz")

reactant.info.update({"charge": SN2_CHARGE, "spin": SN2_MULTIPLICITY})
product.info.update({"charge": SN2_CHARGE, "spin": SN2_MULTIPLICITY})

print("Reactant:", reactant)
print("Product:", product)

_, aligned_product = project_trans_rot(
    reactant.get_positions(), product.get_positions()
)
product.set_positions(aligned_product.reshape(-1, 3))

We are now ready to do the FSM calculation itself, using the default settings.

In [ ]:
string = FreezingString(reactant, product)
optimizer = CartesianOptimizer(calc_polar_1_s)

i = 1
while string.growing:
    string.grow()
    string.optimize(optimizer)
    print(f"Frame {i}")
    i += 1

After the calculation is done, we can extract the energies and visualize them along the string of the Freezing String Method. This should be an approximation of the reaction coordinate, with the approximate TS as the highest point along the path.

In [ ]:
# Extract the path and energies
all_atoms = string.r_string + string.p_string[::-1]
all_energies = np.array(string.r_energy + string.p_energy[::-1])
all_energies = all_energies - min(all_energies)
all_energies_kcal = all_energies * EV_TO_KCAL_MOL
ts_idx = all_energies.argmax()
path = [structure.get_positions() for structure in all_atoms]
s = calculate_arc_length(np.array(path))

# Take out and save path and guess TS coordinates
ts_guess = all_atoms[ts_idx]
ts_guess_xyz = SN2_OUTPUT_DIR / "ts_guess.xyz"
write(ts_guess_xyz, ts_guess)
print(f"Saved TS guess: {ts_guess_xyz}")

fsm_path_xyz = SN2_OUTPUT_DIR / "fsm_path.xyz"
write(fsm_path_xyz, all_atoms)
print(f"Saved FSM path: {fsm_path_xyz} ({len(all_atoms)} frames)")

# Plot
fig, ax = plt.subplots()
ax.plot(s, all_energies_kcal, label="FSM Path")
ax.scatter(s[ts_idx], all_energies_kcal[ts_idx], color="C1", label="TS Guess", zorder=2)
ax.set_xlabel("Arclength (Å)")
ax.set_ylabel("Energy (kcal/mol)")
_ = ax.legend()

Let's look at the structures found in 3D. 

In [ ]:
# Example usage
frames_to_show = [all_atoms[0], all_atoms[ts_idx], all_atoms[-1]]
custom_labels = [
    "Reactant complex<br> Relative Energy: {:.1f} kcal/mol".format(
        all_energies_kcal[0]
    ),
    "TS Guess<br> Relative Energy: {:.1f} kcal/mol".format(all_energies_kcal[ts_idx]),
    "Product complex<br> Relative Energy: {:.1f} kcal/mol".format(
        all_energies_kcal[-1]
    ),
]
snapshot_row(frames_to_show, custom_labels)

## Transition state optimization

We will now go on with the full transition state optimization

In [ ]:
ts_opt = ts_guess.copy()
ts_opt.calc = calc_polar_1_s
ts_opt_traj = SN2_OUTPUT_DIR / "ts_opt.traj"
opt = Sella(ts_opt, trajectory=str(ts_opt_traj), internal=False)
opt.run(fmax=0.02)

ts_opt_xyz = SN2_OUTPUT_DIR / "ts_opt.xyz"
write(ts_opt_xyz, ts_opt)
print(f"Saved TS optimized: {ts_opt_xyz}")

We can visualize the guess and the optimized TS structures.

In [ ]:
def _ts_label(name, atoms):
    r_cf = atoms.get_distance(C_IDX, F_IDX)
    r_ccl = atoms.get_distance(C_IDX, CL_IDX)
    return f"{name}<br>C-F: {r_cf:.2f} Å, C-Cl: {r_ccl:.2f} Å"


snapshot_row(
    [ts_guess, ts_opt],
    [
        _ts_label("TS guess", ts_guess),
        _ts_label("TS optimized", ts_opt),
    ],
)

## 2D potential energy surface

The S<sub>N</sub>2 reaction F⁻ + CH₃Cl → CH₃F + Cl⁻ is governed by two coupled bonds: the **forming** C–F bond and the **breaking** C–Cl bond. Scanning both distances on a grid — and relaxing every other degree of freedom at each point — maps out the full reaction surface (a [More O'Ferrall–Jencks](https://en.wikipedia.org/wiki/More_O%27Ferrall–Jencks_plot) plot).

The reactant and product sit in opposite valleys; the minimum-energy path between them climbs over a saddle point — the transition state we located above with the freezing-string method and Sella. Overlaying that path and the optimized TS on the surface shows them threading through the saddle.

We will first calculate the 2D surface, which will take around one minute. If you have access to a GPU, you can increase the number of grid points for a finer accuracy. (Some points on the grid might take longer to optimize than others.)

In [ ]:
N_GRID = 8
cf_vals = np.linspace(2.6, 1.3, N_GRID)  # forming C-F bond
ccl_vals = np.linspace(1.6, 3.4, N_GRID)  # breaking C-Cl bond

E_grid, optimized_grid, prior_structures = relaxed_2d_scan(
    seed_atoms=reactant,
    x_vals=cf_vals,
    y_vals=ccl_vals,
    x_bond=(C_IDX, F_IDX),
    y_bond=(C_IDX, CL_IDX),
    calculator=calc_polar_1_s,
    charge=SN2_CHARGE,
    spin=SN2_MULTIPLICITY,
    fmax=0.05,
    steps=200,
    col_label="C-F",
    row_label="C-Cl",
)

print(f"\nSurface spans 0 - {E_grid.max() * EV_TO_KCAL_MOL:.2f} kcal/mol")

Now that we have a 2D surface, we can show the guess TS, the optimized TS and the FSM path and judge how they relate to the topology of the surface. We will also compare it to the DFT transition state optimized at the same level of DFT as the machine learning potentials MACE-POLAR-1-S and OrbMol were trained on.

In [ ]:
# Get the bond distance along the FSM path
r_cf_path = [a.get_distance(C_IDX, F_IDX) for a in all_atoms]
r_ccl_path = [a.get_distance(C_IDX, CL_IDX) for a in all_atoms]

# Take out bond distances at the FSM TS guess (string maximum)
guess_cf, guess_ccl = all_atoms[ts_idx].get_distance(C_IDX, F_IDX), all_atoms[
    ts_idx
].get_distance(C_IDX, CL_IDX)

# Take out bond distances at the optimized TS
ts_cf, ts_ccl = ts_opt.get_distance(C_IDX, F_IDX), ts_opt.get_distance(C_IDX, CL_IDX)

# Read the reference DFT TS and take out bond distances
ts_dft = read(SN2_DATA_DIR / "ts_dft.xyz")
dft_cf = ts_dft.get_distance(C_IDX, F_IDX)
dft_ccl = ts_dft.get_distance(C_IDX, CL_IDX)

# Cap the colour scale so the reaction channel and barrier stay visible
fig, ax = plt.subplots(figsize=(7, 6))
draw_pes_meshgrid(
    fig,
    ax,
    cf_vals,
    ccl_vals,
    E_grid,
    colorbar_label="Relative energy (kcal/mol)",
    vmax_cap=3.0,
)

# Minimum-energy path from the freezing-string method, plus TS guess and TS.
ax.plot(r_cf_path, r_ccl_path, "o-", label="FSM path")
ax.scatter(
    [guess_cf],
    [guess_ccl],
    edgecolor="k",
    s=120,
    marker="D",
    label="FSM TS guess",
    zorder=6,
)
ax.scatter(
    [ts_cf],
    [ts_ccl],
    edgecolor="k",
    s=120,
    marker="*",
    label="Optimized TS",
    zorder=7,
)
ax.scatter(
    [dft_cf],
    [dft_ccl],
    edgecolor="k",
    s=120,
    marker="^",
    label="DFT TS",
    zorder=8,
)

ax.set_xlabel("Forming C–F distance (Å)")
ax.set_ylabel("Breaking C–Cl distance (Å)")
ax.set_title("S$_N$2 relaxed 2D potential energy surface")
ax.legend(loc="upper right")
plt.tight_layout()


> How well does the FSM guess compare to the optimized TS?

> How well does the optimized TS with the MLP compare to DFT?

## Comparison between different MLP models

We will now do the comparison between the different MLP models mentioned in the introduction. We will not run the whole FSM procedure, but just optimize the TS structures with the previously optimized TS as a guess. If you have not already run the `test_environment.ipynb` notebook, it might take some time to download the models the first time that you run this.

In [ ]:
# Reoptimize the same TS guess with four different models.
orbff, atoms_adapter = pretrained.orb_v3_conservative_omol(
    device=DEVICE,
    precision="float64",
)

# OrbMol
calc_orbmol = ORBCalculator(orbff, atoms_adapter=atoms_adapter, device=DEVICE)
ts_orb, e_orb = reoptimize_ts(
    ts_opt.copy(),
    calc_orbmol,
    str(SN2_OUTPUT_DIR / "ts_opt_orb_v3_conservative_omol.traj"),
    "TS (ORB-v3-conservative-omol)",
    charge=SN2_CHARGE,
    spin=SN2_MULTIPLICITY,
)

# MACE-Polar-1-M, a larger version of MACE-POLAR-1-S
calc_polar_1_m = mace_polar(model="polar-1-m", default_dtype="float64", device=DEVICE)
ts_polar_1_m, e_polar_1_m = reoptimize_ts(
    ts_opt.copy(),
    calc_polar_1_m,
    str(SN2_OUTPUT_DIR / "ts_opt_polar_1m.traj"),
    "TS (MACE-Polar-1-m)",
    charge=SN2_CHARGE,
    spin=SN2_MULTIPLICITY,
)

# MACE-MP-0
calc_mp = mace_mp(model="small", default_dtype="float64", device=DEVICE)
ts_mp, e_mp = reoptimize_ts(
    ts_opt.copy(),
    calc_mp,
    str(SN2_OUTPUT_DIR / "ts_opt_mace_mp.traj"),
    "TS (MACE-MP-0)",
    charge=SN2_CHARGE,
    spin=SN2_MULTIPLICITY,
)

# MACE-OFF
calc_off = mace_off(model="small", default_dtype="float64", device=DEVICE)
ts_off, e_off = reoptimize_ts(
    ts_mp.copy(),  # Use the MACE-MP-0 optimized TS as the starting point for MACE-OFF, since it's more similar to the final MACE-OFF TS than the original guess is. This helps convergence.
    calc_off,
    str(SN2_OUTPUT_DIR / "ts_opt_mace_off.traj"),
    "TS (MACE-OFF)",
    charge=SN2_CHARGE,
    spin=SN2_MULTIPLICITY,
)

We are now ready to compare the TS positions of all the different methods in the same plot.

In [ ]:
# New 2D PES plot with reoptimized TS structures overlaid
fig, ax = plt.subplots(figsize=(7, 6))
draw_pes_meshgrid(
    fig,
    ax,
    cf_vals,
    ccl_vals,
    E_grid,
    colorbar_label="Relative energy (kcal/mol)",
    vmax_cap=3.0,
    levels_count=25,
)

# Use one marker size for every TS point in this overlay.
ts_marker_size = 120

# Show the baseline TS from the original optimization.
base_cf = ts_opt.get_distance(C_IDX, F_IDX)
base_ccl = ts_opt.get_distance(C_IDX, CL_IDX)
ax.scatter(
    [base_cf],
    [base_ccl],
    edgecolor="k",
    s=ts_marker_size,
    marker="*",
    label="TS (baseline)",
    zorder=10,
)

# Add DFT TS for direct comparison in the same coordinate space.
ts_dft = read(SN2_DATA_DIR / "ts_dft.xyz")
dft_cf = ts_dft.get_distance(C_IDX, F_IDX)
dft_ccl = ts_dft.get_distance(C_IDX, CL_IDX)
ax.scatter(
    [dft_cf],
    [dft_ccl],
    edgecolor="k",
    s=ts_marker_size,
    marker="^",
    label="TS (DFT)",
)

# Overlay reoptimized TS structures currently in use.
ts_points = [
    ("TS (ORB-v3-conservative-omol)", ts_orb, "o"),
    ("TS (MACE-OFF)", ts_off, "P"),
    ("TS (MACE-MP-0)", ts_mp, "D"),
    ("TS (MACE-Polar-1-m)", ts_polar_1_m, "X"),
]

for label, atoms, marker in ts_points:
    r_cf = atoms.get_distance(C_IDX, F_IDX)
    r_ccl = atoms.get_distance(C_IDX, CL_IDX)
    ax.scatter(
        [r_cf],
        [r_ccl],
        edgecolor="k",
        s=ts_marker_size,
        marker=marker,
        zorder=9,
        label=label,
    )

ax.set_xlabel("Forming C-F distance (A)")
ax.set_ylabel("Breaking C-Cl distance (A)")
ax.set_title("SN2 2D PES with reoptimized TS structures")
ax.legend(loc="upper right")
plt.tight_layout()

> How do they compare? Which one is better? How can we know which one is "better"?

We will now compare the intrinsic reaction coordinates (IRCs) for three of the MLPs: MACE-Polar-1-S, OrbMol and MACE-OFF. This will give us an indication of how the different methods differ in their view not only of the placement of the TS, but the overall potential energy surface and the reaction path. IRC calculations are often quite slow, but will go fast for this small reaction.

In [ ]:
# Run IRCs for all SN2 TS models and store everything in one dictionary.
irc_jobs = [
    ("orb", "ORB", "C0", ts_orb, SN2_OUTPUT_DIR / "irc_orb"),
    ("off", "MACE-OFF", "C1", ts_off, SN2_OUTPUT_DIR / "irc_mace_off"),
    (
        "polar",
        "MACE-Polar-1-S",
        "C2",
        ts_opt,
        SN2_OUTPUT_DIR / "irc_polar_1s",
    ),
]

irc_results = {}
for key, label, color, ts_atoms, prefix in irc_jobs:
    images = ordered_irc_path(
        ts_atoms=ts_atoms,
        prefix=prefix,
        charge=SN2_CHARGE,
        spin=SN2_MULTIPLICITY,
        fmax=0.005,
    )
    irc_results[key] = {
        "label": label,
        "color": color,
        "images": images,
        "cf": [a.get_distance(C_IDX, F_IDX) for a in images],
        "ccl": [a.get_distance(C_IDX, CL_IDX) for a in images],
        "count": len(images),
    }
    print(f"Prepared IRC path ({label}): {irc_results[key]['count']} points")

print(
    f"Prepared IRC paths. ORB points: {irc_results['orb']['count']}, "
    f"MACE-OFF points: {irc_results['off']['count']}, "
    f"MACE-Polar-1-s points: {irc_results['polar']['count']}"
)

Now that we have calculated the IRCs, we can display them on top of the 2D potential energy surface.

In [ ]:
# Draw ORB, MACE-OFF, and MACE-Polar-1-s IRC paths on the polar-1-s PES.
fig, ax = plt.subplots(figsize=(7, 6))
_, _, _ = draw_pes_meshgrid(
    fig,
    ax,
    cf_vals,
    ccl_vals,
    E_grid,
    colorbar_label="Relative energy (kcal/mol) [MACE-Polar-1-S PES]",
)

for key in ("orb", "off", "polar"):
    result = irc_results[key]
    ax.plot(
        result["cf"],
        result["ccl"],
        color=result["color"],
        lw=1.8,
        label=f"IRC path ({result['label']})",
    )

# Plot TS structures directly in this cell (removed utility wrapper).
ax.scatter(
    [ts_orb.get_distance(C_IDX, F_IDX)],
    [ts_orb.get_distance(C_IDX, CL_IDX)],
    color="C0",
    edgecolor="k",
    s=120,
    marker="o",
    label="TS (ORB)",
    zorder=8,
)

ax.scatter(
    [ts_off.get_distance(C_IDX, F_IDX)],
    [ts_off.get_distance(C_IDX, CL_IDX)],
    color="C1",
    edgecolor="k",
    s=120,
    marker="*",
    label="TS (MACE-OFF)",
    zorder=9,
)

ax.scatter(
    [ts_opt.get_distance(C_IDX, F_IDX)],
    [ts_opt.get_distance(C_IDX, CL_IDX)],
    color="C2",
    edgecolor="k",
    s=120,
    marker="*",
    label="TS (MACE-Polar-1-s)",
    zorder=9,
)


# Add DFT TS for direct reference in the same coordinate space.
ts_dft = read(SN2_DATA_DIR / "ts_dft.xyz")
ax.scatter(
    [ts_dft.get_distance(C_IDX, F_IDX)],
    [ts_dft.get_distance(C_IDX, CL_IDX)],
    edgecolor="k",
    color="C3",
    s=120,
    marker="^",
    label="TS (DFT)",
    zorder=9,
)

ax.set_xlabel("Forming C-F distance (A)")
ax.set_ylabel("Breaking C-Cl distance (A)")
ax.set_title("ORB, MACE-OFF and MACE-Polar-1-S IRC on polar-1-S 2D PES")
ax.legend(loc="best")
plt.tight_layout()

print(f"ORB IRC points: {irc_results['orb']['count']}")
print(f"MACE-OFF IRC points: {irc_results['off']['count']}")
print(f"MACE-Polar-1-S IRC points: {irc_results['polar']['count']}")

Look at the transition states below, and the IRCs above. 

> What is the agreement between MACE-Polar-1-S and MACE-OFF and OrbMol? Compared to DFT? What could be the reason for these differences?

In [ ]:
# Show the TS structures used for the IRC comparison.
def _ts_label(name, atoms):
    r_cf = atoms.get_distance(C_IDX, F_IDX)
    r_ccl = atoms.get_distance(C_IDX, CL_IDX)
    return f"{name}<br>C-F: {r_cf:.2f} A, C-Cl: {r_ccl:.2f} A"


snapshot_row(
    [ts_opt, ts_off, ts_orb],
    [
        _ts_label("TS (MACE-Polar-1-s)", ts_opt),
        _ts_label("TS (MACE-OFF)", ts_off),
        _ts_label("TS (ORB)", ts_orb),
    ],
)

# Oxidative addition to Pd

After having looked at a reaction that can be challenging as it contains charged species, we will now investigate another potentially difficult reaction. This time it contains metals (Pd) and other heavy elements (Br): The oxidative addition of bromobenzene to a Pd(0) complex. Let's set up the folders, atom indices and charges and multiplicity.

![Oxidative addition](data/drawings/oxidative_addition.png)

We'll use the OrbMol model as it is a bit faster than MACE-Polar, so that you can hopefully finish the calculation even on a slower computer.

In [ ]:
OA_DATA_DIR = Path("data") / "oxidative_addition"
OA_OUTPUT_DIR = Path("output") / "oxidative_addition"
OA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

C_IDX, PD_IDX, BR_IDX = 0, 3, 1

CHARGE_OA = 0
MULTIPLICITY_OA = 1

## Freezing string method

We will start with the generation of the approximate path and guess transition state using the FSM method. The procedure is the same as for the S<sub>N</sub>2 reaction above but will be much slower as the system is larger.

In [ ]:
reactant_complex = read(OA_DATA_DIR / "reactant_complex.xyz")
product_complex = read(OA_DATA_DIR / "product_complex.xyz")

for a in (reactant_complex, product_complex):
    a.info.update({"charge": CHARGE_OA, "spin": MULTIPLICITY_OA})

# Align the product onto the reactant frame (remove rigid translation/rotation).
_, aligned_product_oa = project_trans_rot(
    reactant_complex.get_positions(), product_complex.get_positions()
)
product_complex.set_positions(aligned_product_oa.reshape(-1, 3))

# Reuse the prebuilt calculator for oxidative addition.
calc_oa = calc_orbmol

# Freezing-string method: grow a string between the two endpoints.
string_oa = FreezingString(reactant_complex.copy(), product_complex.copy())
optimizer_oa = CartesianOptimizer(calc_oa)

i = 1
while string_oa.growing:
    string_oa.grow()
    string_oa.optimize(optimizer_oa)
    print(f"Bromobenzene FSM frame {i}")
    i += 1

# Extract energies and reaction path
all_atoms_oa = string_oa.r_string + string_oa.p_string[::-1]
all_energies_oa = np.array(string_oa.r_energy + string_oa.p_energy[::-1])
all_energies_oa -= all_energies_oa.min()
ts_idx_oa = all_energies_oa.argmax()
s_oa = calculate_arc_length(
    np.array([struct.get_positions() for struct in all_atoms_oa])
)

# Save the full FSM path as a multistructure XYZ trajectory.
fsm_path_xyz = OA_OUTPUT_DIR / "fsm_oa_path.xyz"
write(fsm_path_xyz, all_atoms_oa)
print(f"Saved FSM path: {fsm_path_xyz} ({len(all_atoms_oa)} frames)")

We can now plot the reaction path

In [ ]:
all_energies_oa_kcal = all_energies_oa * EV_TO_KCAL_MOL

fig, ax = plt.subplots()
ax.plot(s_oa, all_energies_oa_kcal, "-o", ms=3, label="FSM path (polar-1-s)")
ax.scatter(
    s_oa[ts_idx_oa],
    all_energies_oa_kcal[ts_idx_oa],
    color="C1",
    label="TS guess",
    zorder=2,
)
ax.set_xlabel("Arclength (A)")
ax.set_ylabel("Relative energy (kcal/mol)")
ax.set_title("Bromobenzene FSM scan (MACE-Polar-1S)")
_ = ax.legend()

print(
    f"FSM done: {len(all_atoms_oa)} images, "
    f"barrier ~ {all_energies_oa_kcal[ts_idx_oa]:.3f} kcal/mol"
)

## Transition state optimization

We will now optimize the transition state starting from the guess obtained from the FSM. This should take less than a minute.

In [ ]:
ts_guess_oa = all_atoms_oa[ts_idx_oa].copy()
ts_guess_oa.calc = calc_oa

ts_oa_traj = OA_OUTPUT_DIR / "ts_oa_from_fsm.traj"
opt_ts_oa = Sella(
    ts_guess_oa,
    trajectory=str(ts_oa_traj),
    internal=False,
)
opt_ts_oa.run(fmax=0.02)

e_ts_oa = ts_guess_oa.get_potential_energy()
print(
    f"Oxidative addition TS (MACE-Polar-1-S, from FSM guess): "
    f"{e_ts_oa * EV_TO_KCAL_MOL:.6f} kcal/mol"
)

Let's compare the two transition states.

In [ ]:
ts_oa_traj = OA_OUTPUT_DIR / "ts_oa_from_fsm.traj"
ts_guess_oa_struct = read(ts_oa_traj, index=0)
ts_oa_optimized_struct = read(ts_oa_traj, index=-1)

ts_oa_optimized_xyz = OA_OUTPUT_DIR / "ts_oa_optimized.xyz"
write(ts_oa_optimized_xyz, ts_oa_optimized_struct)
print(f"Saved optimized OA TS to {ts_oa_optimized_xyz}")


def _oa_ts_label(name, atoms):
    r_cpd = atoms.get_distance(C_IDX, PD_IDX)
    r_cbr = atoms.get_distance(C_IDX, BR_IDX)
    r_pdbr = atoms.get_distance(PD_IDX, BR_IDX)
    return f"{name}<br>C-Pd: {r_cpd:.2f} Å, C-Br: {r_cbr:.2f} Å, Pd-Br: {r_pdbr:.2f} Å"


snapshot_row(
    [ts_guess_oa_struct, ts_oa_optimized_struct],
    [
        _oa_ts_label("TS guess (OA)", ts_guess_oa_struct),
        _oa_ts_label("TS optimized (OA)", ts_oa_optimized_struct),
    ],
)

> How similar are they to each other in terms of the reactive bonds? How do you think this affects the number of optimization steps as seen above? 

## IRC
Now we will run the IRC. To save time, we use larger step sizes (`dx`) and fewer steps (`steps`). You can change this if you have GPU acceleration. The IRC should finish within a couple of minutes.

In [ ]:
fwd_traj = OA_OUTPUT_DIR / "irc_oa_forward.traj"
rev_traj = OA_OUTPUT_DIR / "irc_oa_reverse.traj"
irc_steps = 20
irc_dx = 0.25

irc_ts = opt_ts_oa.atoms.copy()
irc_ts.calc = calc_oa
irc_images = ordered_irc_path(
    irc_ts,
    str(OA_OUTPUT_DIR / "irc_oa"),
    charge=CHARGE_OA,
    spin=MULTIPLICITY_OA,
    dx=irc_dx,
    fmax=0.01,
    steps=irc_steps,
    keep_going=True,
)

irc_xyz = OA_OUTPUT_DIR / "irc_oa_path.xyz"
write(irc_xyz, irc_images)

fwd_images = read(fwd_traj, index=":")
rev_images = read(rev_traj, index=":")

print(f"Saved IRC path: {irc_xyz} ({len(irc_images)} frames)")
print(f"IRC forward images (oxidative addition): {len(fwd_images)}")
print(f"IRC reverse images (oxidative addition): {len(rev_images)}")
print(f"Total stitched IRC points (oxidative addition): {len(irc_images)}")

We first plot the IRC with the energy as a function of teh reaction coordinate.

In [ ]:
# Take out energies
irc_images = list(reversed(rev_images)) + fwd_images[1:]
irc_energies = np.array([atoms.calc.results.get("energy") for atoms in irc_images])
irc_energies_rel = irc_energies - irc_energies.min()
irc_energies_rel_kcal = irc_energies_rel * EV_TO_KCAL_MOL

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    np.arange(len(irc_energies_rel_kcal)),
    irc_energies_rel_kcal,
    "-o",
    label="IRC (oxidative addition, OrbMol)",
)
ax.set(
    xlabel="IRC point index",
    ylabel="Relative energy (kcal/mol)",
)
ax.legend(loc="best")
plt.tight_layout()

print(f"Forward images: {len(fwd_images)}")
print(f"Reverse images: {len(rev_images)}")
print(f"Total stitched IRC points: {len(irc_images)}")

## 2D potential energy surface

Now we will calculate the 2D potential energy surface and overlay the FSM, IRC and TS structures. To save time, we will only calculate a part of the surface, and also with a smaller number of grid points and with less strict convergence criteria. If you have GPU acceleration, you can increase the limits and and number of grid points. This calculation will take at least a few minutes.

In [ ]:
scan_seed = reactant_complex.copy()
scan_seed.calc = calc_oa

N_GRID_PD = 5
cpd_vals = np.linspace(2.5, 1.8, N_GRID_PD)
cbr_vals = np.linspace(1.8, 2.5, N_GRID_PD)

E_grid_pd, optimized_grid_pd, prior_structures_pd = relaxed_2d_scan(
    seed_atoms=scan_seed,
    x_vals=cpd_vals,
    y_vals=cbr_vals,
    x_bond=(C_IDX, PD_IDX),
    y_bond=(C_IDX, BR_IDX),
    calculator=calc_oa,
    charge=CHARGE_OA,
    spin=MULTIPLICITY_OA,
    fmax=0.1,
    steps=120,
    col_label="C-Pd",
    row_label="C-Br",
)

Now we will draw the IRC, FSM, guess and optimized transition states on the 2D potential energy surface.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
draw_pes_meshgrid(
    fig,
    ax,
    cpd_vals,
    cbr_vals,
    E_grid_pd,
    colorbar_label="Relative energy (kcal/mol)",
)

# Overlay FSM and IRC paths in the same reaction coordinates as the PES.
fsm_cpd = [a.get_distance(C_IDX, PD_IDX) for a in all_atoms_oa]
fsm_cbr = [a.get_distance(C_IDX, BR_IDX) for a in all_atoms_oa]
irc_cpd = [a.get_distance(C_IDX, PD_IDX) for a in irc_images]
irc_cbr = [a.get_distance(C_IDX, BR_IDX) for a in irc_images]
ax.plot(fsm_cpd, fsm_cbr, "-o", lw=1.8, label="FSM path", zorder=6)
ax.plot(irc_cpd, irc_cbr, "-o", lw=1.8, label="IRC path", zorder=7)

# Mark TS guess and optimized TS.
ts_guess_cpd = ts_guess_oa.get_distance(C_IDX, PD_IDX)
ts_guess_cbr = ts_guess_oa.get_distance(C_IDX, BR_IDX)
ts_opt_cpd = ts_guess_oa_struct.get_distance(C_IDX, PD_IDX)
ts_opt_cbr = ts_guess_oa_struct.get_distance(C_IDX, BR_IDX)
ax.scatter(
    [ts_opt_cpd],
    [ts_opt_cbr],
    edgecolor="k",
    s=120,
    marker="*",
    label="TS optimized",
    zorder=8,
)
ax.scatter(
    [ts_guess_cpd],
    [ts_guess_cbr],
    edgecolor="k",
    s=120,
    marker="D",
    label="TS guess",
    zorder=9,
)

ax.set_xlabel("C-Pd distance (A)")
ax.set_ylabel("C-Br distance (A)")
ax.set_title("Relaxed 2D scan with FSM, IRC, and TS structures")
ax.legend(loc="best")
plt.tight_layout()

print(f"C-Pd range: {cpd_vals[0]:.2f} -> {cpd_vals[-1]:.2f} A")
print(f"C-Br range: {cbr_vals[0]:.2f} -> {cbr_vals[-1]:.2f} A")
print(f"Surface span: 0 -> {E_grid_pd.max() * EV_TO_KCAL_MOL:.3f} kcal/mol")
print(f"FSM points: {len(all_atoms_oa)}")
print(f"IRC points: {len(irc_images)}")

> How good is the agreement between the freezing string method and its guess TS and the NEB in this more complicated case? What is the risk if it is too far away? How could you increase the accuracy of the FSM?

> How good is the accuracy compared to DFT?

> Why could we not use MACE-OFF to study this reaction?

## Bonus task

Now it is time to move on to the notebook on descriptor calculation with MLIPs: `descriptors.ipynb`.

Or you can take on the challenge to model the 1,3-dipolar cycloaddition reaction between hydrazoic acid and acetylene. You can find the structures that you need in `data/cycloaddition`. You need to determine the indices of the reactive atoms of interest, and write your own code to do your calculations and analysis.
